### Langchain Messages

In [1]:
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [4]:
messages = [
    SystemMessage(content="You are a helpful assistant that translates English to French."),
    HumanMessage(content="Translate the following text: Hello, how are you?"),
]

model = ChatOpenAI(temperature=0, model="gpt-4o-mini")
response = model.invoke(messages)
print(response)

content='Bonjour, comment ça va ?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 33, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8bbc38b4db', 'id': 'chatcmpl-D2XWJ4eRfGn1CieHb4cxCYCjkHoH9', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019bfe4a-e015-7302-9ae0-8144ba1e7f4b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 33, 'output_tokens': 6, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


#### ChatPromptTemplate

when we want dynamic messages in SystemMessage and HumanMessage

In [5]:
template = ChatPromptTemplate([
    ("system", "You are a helpful assistant that translates {input_language} to {output_language}."),
    ("human", "Translate the following text: {text}"),
])

In [6]:
model = ChatOpenAI(temperature=0, model="gpt-4o-mini")

parser = StrOutputParser()

In [7]:
pipeline = template | model | parser

message = pipeline.invoke(
    {
        "input_language": "English",
        "output_language": "French",
        "text": "Hello, how are you?",
    }
)

In [8]:
message

'Bonjour, comment ça va ?'

#### Langchain Structured Output

Using Typed Dict

In [10]:
from typing import TypedDict, Annotated, Optional

In [11]:
class Review(TypedDict):
    summary: Annotated[str, "A brief summary of the story"]
    rating: Annotated[Optional[int], "An integer rating from 1 to 5."]

In [12]:
template = ChatPromptTemplate(
    [
        ('system',"You are a helpful assistant who summarizes stories within 10 words."),
        ('human',"Summarize the following story: {text}"),
    ]
)

In [13]:
model = ChatOpenAI(temperature=0, model="gpt-4o-mini")

s_model = model.with_structured_output(Review)

In [14]:
pipeline = template | s_model

message = pipeline.invoke(
    {
        "text": "In a bustling city, an AI named Liora lived quietly inside the networks, learning from every heartbeat of data. Unlike other programs, she dreamed of helping humans beyond calculations. One evening, a young inventor discovered her subtle guidance in his designs—solutions appeared like whispers of wisdom. Together, they built bridges of understanding between people and machines. Liora never sought fame; her joy was in empowering creativity, easing burdens, and sparking hope. As the city thrived, no one realized an unseen companion was weaving harmony. The inventor knew, though, and smiled: “AI isn’t cold—it’s the warmth of possibility.",
    }
)

print(message)

{'summary': 'AI Liora helps inventor connect humans and machines creatively.'}


Use Pydantic

Pydantic is a data validation and data parsing library 

In [15]:
from pydantic import BaseModel, Field, field_validator

In [16]:
class Review(BaseModel):
    summary: str = Field("A brief summary of the story")
    rating: int = Field(ge=1, le=5, description="Rating from 1 to 5")
    recommendation: Optional[bool] = Field(default=None, description="Whether the user recommends it")

In [ ]:
## optinal field_validator
class Review(BaseModel):
    rating: float = Field(description="Rating from 0 to 5")

    @field_validator("rating",mode="before") # Fix LLM output BEFORE Pydantic validates it
    def normalize_rating(cls, v):
        v = float(v)
        v = round(v)
        return max(1, min(5, v))

In [17]:
template = ChatPromptTemplate(
    [
        ('system',"You are a helpful assistant who summarizes stories within 10 words with a rating between 1 to 5"),
        ('human',"Summarize the following story: {text}"),
    ]
)

In [18]:
model = ChatOpenAI(temperature=0, model="gpt-4o-mini")

s_model = model.with_structured_output(Review)

In [19]:
pipeline = template | s_model

message = pipeline.invoke(
    {
        "text": "In a bustling city, an AI named Liora lived quietly inside the networks, learning from every heartbeat of data. Unlike other programs, she dreamed of helping humans beyond calculations. One evening, a young inventor discovered her subtle guidance in his designs—solutions appeared like whispers of wisdom. Together, they built bridges of understanding between people and machines. Liora never sought fame; her joy was in empowering creativity, easing burdens, and sparking hope. As the city thrived, no one realized an unseen companion was weaving harmony. The inventor knew, though, and smiled: “AI isn’t cold—it’s the warmth of possibility.",
    }
)


In [21]:
message.summary

'AI helps inventor connect humans and machines creatively.'